In [ ]:
pip install pandas matplotlib openpyxl

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import numpy as np
import seaborn as sns
warnings.filterwarnings('ignore')

plt.rcParams["font.family"] = ["PingFang SC", "Arial Unicode MS", "Heiti TC"]
plt.rcParams["axes.unicode_minus"] = False

file_path = "data/processed.xlsx"

df_main = pd.read_excel(file_path, sheet_name="3 provinces")
df_sub = pd.read_excel(file_path, sheet_name="type")

df_sub_clean = df_sub.dropna(subset=["ID", "Abstract"]).copy()
df_type = df_sub.groupby("ID")["Type"].apply(list).reset_index()
df_type = df_type.explode("Type").dropna(subset=["Type"])

df_final = pd.merge(df_main, df_type, on="ID", how="inner")

df_final["Year"] = pd.to_numeric(df_final["Year"], errors="coerce")
df_year_valid = df_final.dropna(subset=["Year"])

df_op_valid = df_final.dropna(subset=["Core_OP"])

df_loc_valid = df_final.dropna(subset=["Location"])


def plot_type_pie():
    type_count = df_final["Type"].value_counts()

    threshold = 2
    mask = type_count / type_count.sum() * 100 < threshold
    small_categories = type_count[mask]
    big_categories = type_count[~mask]

    if not small_categories.empty:
        big_categories["Others"] = small_categories.sum()

    plt.figure(figsize=(9, 9))

    wedges, texts, autotexts = plt.pie(
        big_categories,
        labels=big_categories.index,
        autopct="%1.1f%%",
        startangle=90,
        pctdistance=0.85,
        labeldistance=1.1,
        wedgeprops=dict(width=0.3),
        textprops=dict(fontsize=11)
    )

    for text in texts:
        text.set_horizontalalignment('center')
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(10)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    plot_type_pie()


In [ ]:
# ===================== Figure 3-2: Heatmap of the Correlation Strength of Memorial Types =====================

plt.rcParams["font.family"] = ["PingFang SC", "Heiti TC", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

df_sub = pd.read_excel(file_path, sheet_name="type")
df_sub["ID"] = df_sub["ID"].ffill()
df_type_clean = df_sub.dropna(subset=["ID", "Type"]).drop_duplicates(subset=["ID", "Type"])
type_groups = df_type_clean.groupby("ID")["Type"].apply(list).reset_index()
multi_type_count = sum(1 for types in type_groups["Type"] if len(types) > 1)
all_types = sorted(df_type_clean["Type"].unique())
n = len(all_types)

cooccurrence_matrix = np.zeros((n, n), dtype=int)

for types in type_groups["Type"]:
    for i in range(len(types)):
        for j in range(len(types)):
            idx_i = all_types.index(types[i])
            idx_j = all_types.index(types[j])
            cooccurrence_matrix[idx_i, idx_j] += 1

co_df = pd.DataFrame(cooccurrence_matrix, index=all_types, columns=all_types)

type_paper_count = {t: sum(1 for types in type_groups["Type"] if t in types) for t in all_types}

association_matrix = co_df.copy().astype(float)
for t1 in all_types:
    for t2 in all_types:
        if type_paper_count[t1] > 0:
            association_matrix.loc[t1, t2] = co_df.loc[t1, t2] / type_paper_count[t1]

plt.figure(figsize=(10, 8))
sns.heatmap(
    association_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    vmin=0,
    vmax=1
)
plt.xlabel("Type", fontsize=12)
plt.ylabel("Type", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ===================== Figure 3-3: Overall Temporal Trend of City Wall Memorials =====================

def plot_overall_and_selected_trends():
    min_year = int(df_year_valid["Year"].min())
    max_year = int(df_year_valid["Year"].max())
    full_years = pd.Index(np.arange(min_year, max_year + 1), name="Year")
    
    year_type = df_year_valid.groupby(["Year", "Type"]).size().unstack(fill_value=0)
    year_type = year_type.reindex(full_years, fill_value=0)

    yearly_total = df_year_valid.groupby("Year").size().reindex(full_years, fill_value=0)

    plt.figure(figsize=(14, 4.5))
    plt.bar(yearly_total.index, yearly_total.values, color="#2b5c8f", width=0.8, alpha=0.85)
    plt.xlabel("Year", fontsize=11)
    plt.ylabel("Total Number of Memorials", fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.gca().spines[['top', 'right']].set_visible(False)
    plt.xlim(min_year - 2, max_year + 2)
    plt.tight_layout()
    plt.show()

# ===================== Figure 3-4: Temporal Distribution across Key Event Types =====================

    target_types = ["renovation", "donation", "funding", "inspection", "destruction"]
    
    selected_types = [t for t in target_types if t in year_type.columns]
    
    n_types = len(selected_types)
    fig, axes = plt.subplots(n_types, 1, figsize=(14, 2.2 * n_types), sharex=True, sharey=True)
    
    type_colors = {
        "renovation": "#e377c2",
        "donation": "#2ca02c",
        "funding": "#d62728",
        "inspection": "#9467bd",
        "destruction": "#ff7f0e",
        "construction": "#1f77b4"
    }

    if n_types == 1:
        axes = [axes]

    for ax, t_name in zip(axes, selected_types):
        color = type_colors.get(t_name, "#1f77b4")
        
        ax.plot(year_type.index, year_type[t_name], color=color, linewidth=1.8, label=t_name)
        ax.fill_between(year_type.index, year_type[t_name], color=color, alpha=0.15)
        
        ax.text(0.015, 0.75, t_name.capitalize(), transform=ax.transAxes, 
                fontsize=11, fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.8))
        
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines[['top', 'right']].set_visible(False)

    plt.xlabel("Year", fontsize=11)
    fig.text(-0.01, 0.5, 'Number of Memorials', va='center', rotation='vertical', fontsize=11)
    
    plt.tight_layout()
    plt.show()

plot_overall_and_selected_trends()

In [ ]:
# ===================== Figure 3-5: Typological Distribution of City Wall Memorialized by Officials =====================

def plot_type_op_optimized():

    op_type = pd.crosstab(df_op_valid["Core_OP"], df_op_valid["Type"])
    op_type["Total"] = op_type.sum(axis=1)
    op_type = op_type.sort_values(by="Total", ascending=False).drop(columns=["Total"])
    fig, ax = plt.subplots(figsize=(13, 6))
    colors = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", 
        "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22"
    ]
    
    op_type.plot(kind="bar", stacked=True, ax=ax, color=colors[:len(op_type.columns)], width=0.55)
    

    ax.set_xlabel("Official Title", fontsize=11, fontweight='bold', labelpad=10)
    ax.set_ylabel("Number of Memorials", fontsize=11, fontweight='bold', labelpad=10)
    
    plt.xticks(rotation=35, ha="right", fontsize=9.5)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    
    ax.legend(
        title="Types", 
        bbox_to_anchor=(1.02, 1),
        loc="upper left", 
        frameon=True,
        facecolor="white", 
        edgecolor="none",
        fontsize=10
    )
    
    plt.subplots_adjust(right=0.82, bottom=0.3)
    plt.show()

plot_type_op_optimized()

In [ ]:
# ===================== Figure 3-6: Typological Distribution of City Wall Memorialized by Locations =====================

def plot_type_location_sorted_stacks():
    loc_translation = {
        "山东": "Shandong",
        "四川": "Sichuan",
        "福建": "Fujian",
    }

    df_loc_clean = df_loc_valid.copy()
    df_loc_clean["Location_EN"] = df_loc_clean["Location"].map(loc_translation).fillna(df_loc_clean["Location"])

    loc_type = pd.crosstab(df_loc_clean["Location_EN"], df_loc_clean["Type"])

    loc_type["Total"] = loc_type.sum(axis=1)
    loc_type = loc_type.sort_values(by="Total", ascending=False).drop(columns=["Total"])

    all_types = sorted(df_loc_valid["Type"].unique())
    color_palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", 
        "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22"
    ]
    type_color_map = {t: color_palette[i % len(color_palette)] for i, t in enumerate(all_types)}

    fig, ax = plt.subplots(figsize=(10, 6))
    bar_width = 0.4

    legend_handles = {}

    for x_idx, (loc_name, row_data) in enumerate(loc_type.iterrows()):
        sorted_types = row_data[row_data > 0].sort_values(ascending=False)
        
        bottom = 0 
        for t_name, val in sorted_types.items():
            color = type_color_map[t_name]

            bar = ax.bar(x_idx, val, bottom=bottom, width=bar_width, color=color, edgecolor="white", linewidth=0.5)
            bottom += val

            if t_name not in legend_handles:
                legend_handles[t_name] = bar[0]

    ax.set_xticks(range(len(loc_type)))
    ax.set_xticklabels(loc_type.index, fontsize=11, fontweight="bold")
    ax.set_xlabel("Provinces", fontsize=11, fontweight="bold", labelpad=10)
    ax.set_ylabel("Number of Memorials", fontsize=11, fontweight="bold", labelpad=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    sorted_legend_keys = sorted(legend_handles.keys())
    ax.legend(
        [legend_handles[k] for k in sorted_legend_keys],
        sorted_legend_keys,
        title="Types",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=True,
        facecolor="white",
        edgecolor="none",
        fontsize=10
    )

    plt.subplots_adjust(right=0.82, bottom=0.15)
    plt.show()

plot_type_location_sorted_stacks()

In [ ]:
# ===================== Figure 3-7a: Temporal-Spatial Distribution of City Wall Records =====================

df = df_main

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df.dropna(subset=["Year"])
df["Year"] = df["Year"].astype(int)

loc_translation = {
    "四川": "Sichuan",
    "山东": "Shandong",
    "福建": "Fujian",
}

df["Location_EN"] = df["Location"].map(loc_translation).fillna(df["Location"])

heatmap_data = pd.crosstab(df["Year"], df["Location_EN"])

plt.figure(figsize=(14, 10))
sns.heatmap(
    heatmap_data,
    cmap="OrRd",
    annot=False,
    linewidths=0.5,
    cbar_kws={'label': 'Number of Memorials'}
)

plt.xlabel("Location", fontsize=12, fontweight="bold", labelpad=10)
plt.ylabel("Year", fontsize=12, fontweight="bold", labelpad=10)

plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ===================== Figure 3-7b: Temporal Distribution of city wall affairs: Mainland Fujian vs. Taiwan =====================

df_taiwan = pd.read_excel(file_path, sheet_name="taiwan")

df_sub['ID'] = df_sub['ID'].ffill()
df_sub['Abstract'] = df_sub['Abstract'].ffill()

df = pd.merge(df_taiwan, df_sub[['ID', 'Type']], on='ID', how='left')

df['Region'] = df['Location'].apply(lambda x: 'Taiwan' if '臺灣' in str(x) or 'Taiwan' in str(x) else 'Mainland Fujian')

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(10, 5))
df_unique_id = df.drop_duplicates(subset=['ID'])
df_unique_id['Decade'] = (df_unique_id['Year'] // 10) * 10

timeline_data = df_unique_id.groupby(['Decade', 'Region']).size().reset_index(name='Count')

sns.lineplot(data=timeline_data, x='Decade', y='Count', hue='Region', marker='o', linewidth=2.5)

plt.xlabel("Year (Decade)", fontsize=10)
plt.ylabel("Number of Memorials (Unique IDs)", fontsize=10)
plt.legend(title="Region")
plt.tight_layout()
plt.savefig("Fig1_Temporal_Comparison.png", dpi=300)
plt.show()


In [ ]:
# ===================== Figure 3-8: Stacked Distribution of Officials by Location =====================

loc_translation = {
    "四川": "Sichuan",
    "山东": "Shandong",
    "福建": "Fujian"
}

df = df_main
df["Location_EN"] = df["Location"].map(loc_translation).fillna(df["Location"])
count_data = pd.crosstab(df["Location_EN"], df["Core_OP"])
count_data["Total"] = count_data.sum(axis=1)
count_data = count_data.sort_values(by="Total", ascending=False).drop(columns=["Total"])
fig, ax = plt.subplots(figsize=(10, 6))
colors = [
    "#2e92d8", "#aec7e8", "#ff9437", "#ffbb78", "#46c546", 
    "#98df8a", "#72d2bd", "#ff9896", "#9467bd", "#c5b0d5"
]

count_data.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=colors[:len(count_data.columns)],
    width=0.45
)

ax.set_xlabel("Provinces", fontsize=11, fontweight="bold", labelpad=10)
ax.set_ylabel("Count of Records", fontsize=11, fontweight="bold", labelpad=10)

plt.xticks(rotation=0, fontsize=11, fontweight="bold")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)

ax.legend(
    title="Official Title",
    title_fontsize='11',
    fontsize=9.5,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="none",
    labelspacing=0.6,
    handletextpad=0.5
)

plt.subplots_adjust(right=0.72, bottom=0.15)
plt.show()

In [ ]:
# ===================== Figure 3-9a: Composition of City Wall Affairs (Type): Mainland Fujian vs. Taiwan =====================

df_taiwan = pd.read_excel(file_path, sheet_name="taiwan")
df_sub['ID'] = df_sub['ID'].ffill()
df_sub['Abstract'] = df_sub['Abstract'].ffill()
df = pd.merge(df_taiwan, df_sub[['ID', 'Type']], on='ID', how='left')

df['Region'] = df['Location'].apply(lambda x: 'Taiwan' if '臺灣' in str(x) or 'Taiwan' in str(x) else 'Mainland Fujian')
plt.figure(figsize=(8, 6))
type_counts = pd.crosstab(df['Region'], df['Type'], normalize='index') * 100

ax = type_counts.plot(kind='bar', stacked=True, colormap='Set2', figsize=(8, 6))

plt.xlabel("Region", fontsize=10)
plt.ylabel("Percentage (%)", fontsize=10)
plt.xticks(rotation=0)
plt.legend(title="Affair Type", bbox_to_anchor=(1.05, 1), loc='upper left')

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 3:
        x, y = p.get_xy() 
        ax.text(x + width/2, y + height/2, f'{height:.1f}%', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ===================== Figure 3-9b: Distribution of Memorial Submitting Officials =====================

plt.figure(figsize=(9, 5))

top_ops = df_unique_id['Core_OP'].value_counts().head(5).index
df_top_ops = df_unique_id[df_unique_id['Core_OP'].isin(top_ops)]

op_counts = pd.crosstab(df_top_ops['Region'], df_top_ops['Core_OP'], normalize='index') * 100

op_counts.plot(kind='bar', stacked=False, colormap='Blues', figsize=(9, 5))

plt.xlabel("Region", fontsize=10)
plt.ylabel("Percentage of Memorials (%)", fontsize=10)
plt.xticks(rotation=0)
plt.legend(title="Official Title (Core_OP)", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()